# Notebook 1 — Data Loading & Preprocessing
**Project:** Chest X-ray Pneumonia Detection — Generalization Study 

This notebook:
1. Connects to Kaggle and downloads both datasets
2. Loads images, resizes them, and normalizes pixel values
3. Creates stratified train/test splits
4. Saves preprocessed arrays to disk for use in later notebooks

**Run cells in order, top to bottom.**

## Step 0 — Install & import libraries

In [ ]:
# Install any missing packages
!pip install kaggle opencv-python-headless scikit-learn numpy matplotlib tqdm --quiet

In [ ]:
import os
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Image settings
IMG_SIZE = 64        # resize every image to 64x64 pixels
MAX_NIH  = 2000      # how many NIH images to sample (keeps runtime fast)

print('Libraries loaded successfully.')

## Step 1 — Connect to Kaggle

**One-time setup:**
1. Go to https://www.kaggle.com → Account → API → "Create New Token"
2. This downloads a file called `kaggle.json`
3. Upload that file when the cell below asks for it

In [ ]:
from google.colab import files

print('Upload your kaggle.json file now...')
uploaded = files.upload()   # a file picker will appear

# Move it to the right location so the kaggle CLI can find it
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
kaggle_path = os.path.expanduser('~/.kaggle/kaggle.json')
with open(kaggle_path, 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod(kaggle_path, 0o600)   # required permission

print('kaggle.json saved. Ready to download datasets.')

## Step 2 — Download Dataset 1: Kaggle Chest X-Ray (Pneumonia)

In [ ]:
# Download and unzip (~1.2 GB — takes ~2-3 minutes on Colab)
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia --quiet
print('Download complete. Unzipping...')

with zipfile.ZipFile('chest-xray-pneumonia.zip', 'r') as z:
    z.extractall('data/kaggle')

print('Done! Folder structure:')
for p in sorted(Path('data/kaggle').rglob('*')):
    if p.is_dir():
        count = len(list(p.glob('*.jpeg')) + list(p.glob('*.jpg')) + list(p.glob('*.png')))
        if count > 0:
            print(f'  {p}  ({count} images)')

## Step 3 — Download Dataset 2: NIH Chest X-Ray subset

In [ ]:
# NIH ChestX-ray14 — we use a pre-sampled subset available on Kaggle
!kaggle datasets download -d nih-chest-xrays/data --quiet
print('Download complete. Unzipping (this may take a few minutes)...')

with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('data/nih')

print('NIH dataset ready.')

## Step 4 — Helper function: load and preprocess images

In [ ]:
def load_images_from_folder(folder, label, img_size=IMG_SIZE, max_images=None):
    """
    Load all images from a folder, resize, convert to grayscale,
    and normalize pixel values to [0, 1].

    Returns:
        images : np.array of shape (N, img_size, img_size)
        labels : np.array of shape (N,)  — 0 = Normal, 1 = Pneumonia
    """
    images, labels = [], []
    folder = Path(folder)
    files  = list(folder.glob('*.jpeg')) + list(folder.glob('*.jpg')) + list(folder.glob('*.png'))

    if max_images:
        # Randomly sample to keep things balanced and fast
        rng = np.random.default_rng(SEED)
        files = rng.choice(files, size=min(max_images, len(files)), replace=False).tolist()

    for filepath in tqdm(files, desc=f'Loading {folder.name}'):
        img = cv2.imread(str(filepath), cv2.IMREAD_GRAYSCALE)  # grayscale
        if img is None:
            continue
        img = cv2.resize(img, (img_size, img_size))            # resize
        img = img.astype(np.float32) / 255.0                  # normalize to [0,1]
        images.append(img)
        labels.append(label)

    return np.array(images), np.array(labels)

print('Helper function defined.')

## Step 5 — Load Dataset 1 (Kaggle)

In [ ]:
# Kaggle dataset already has train/test/val splits — we use train + test only
KAGGLE_ROOT = Path('data/kaggle/chest_xray')

# Load training images (Normal=0, Pneumonia=1)
train_normal_imgs, train_normal_labels = load_images_from_folder(
    KAGGLE_ROOT / 'train' / 'NORMAL', label=0)
train_pneum_imgs, train_pneum_labels = load_images_from_folder(
    KAGGLE_ROOT / 'train' / 'PNEUMONIA', label=1)

# Load test images
test_normal_imgs, test_normal_labels = load_images_from_folder(
    KAGGLE_ROOT / 'test' / 'NORMAL', label=0)
test_pneum_imgs, test_pneum_labels = load_images_from_folder(
    KAGGLE_ROOT / 'test' / 'PNEUMONIA', label=1)

# Combine
X_train = np.concatenate([train_normal_imgs, train_pneum_imgs], axis=0)
y_train = np.concatenate([train_normal_labels, train_pneum_labels], axis=0)
X_test  = np.concatenate([test_normal_imgs, test_pneum_imgs], axis=0)
y_test  = np.concatenate([test_normal_labels, test_pneum_labels], axis=0)

print(f'Dataset 1 (Kaggle) loaded:')
print(f'  Training set : {X_train.shape[0]} images  (Normal: {(y_train==0).sum()}, Pneumonia: {(y_train==1).sum()})')
print(f'  Test set     : {X_test.shape[0]} images  (Normal: {(y_test==0).sum()}, Pneumonia: {(y_test==1).sum()})')

## Step 6 — Load Dataset 2 (NIH subset)

In [ ]:
import pandas as pd

# NIH dataset has a CSV with labels — we filter for Normal and Pneumonia only
nih_labels_df = pd.read_csv('data/nih/Data_Entry_2017.csv')

# Keep only Normal and Pneumonia rows
nih_normal = nih_labels_df[nih_labels_df['Finding Labels'] == 'No Finding'].sample(
    n=MAX_NIH // 2, random_state=SEED)
nih_pneum  = nih_labels_df[nih_labels_df['Finding Labels'].str.contains('Pneumonia')].sample(
    n=min(MAX_NIH // 2, nih_labels_df['Finding Labels'].str.contains('Pneumonia').sum()),
    random_state=SEED)

nih_subset = pd.concat([nih_normal, nih_pneum]).reset_index(drop=True)
print(f'NIH subset: {len(nih_normal)} Normal, {len(nih_pneum)} Pneumonia')

# Load the actual image files
NIH_IMG_ROOT = Path('data/nih/images')   # adjust if your path differs
nih_images, nih_label_arr = [], []

for _, row in tqdm(nih_subset.iterrows(), total=len(nih_subset), desc='Loading NIH images'):
    img_path = NIH_IMG_ROOT / row['Image Index']
    if not img_path.exists():
        continue
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    nih_images.append(img)
    label = 1 if 'Pneumonia' in row['Finding Labels'] else 0
    nih_label_arr.append(label)

X_nih = np.array(nih_images)
y_nih = np.array(nih_label_arr)

print(f'\nDataset 2 (NIH) loaded:')
print(f'  {X_nih.shape[0]} images  (Normal: {(y_nih==0).sum()}, Pneumonia: {(y_nih==1).sum()})')

## Step 7 — Verify: visualize sample images from both datasets

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
fig.suptitle('Sample images — top: Kaggle, bottom: NIH', fontsize=13)

# Top row: Kaggle (3 Normal, 3 Pneumonia)
kaggle_normal_idx = np.where(y_train == 0)[0][:3]
kaggle_pneum_idx  = np.where(y_train == 1)[0][:3]
for i, idx in enumerate(list(kaggle_normal_idx) + list(kaggle_pneum_idx)):
    axes[0, i].imshow(X_train[idx], cmap='gray')
    axes[0, i].set_title('Normal' if y_train[idx] == 0 else 'Pneumonia', fontsize=9)
    axes[0, i].axis('off')

# Bottom row: NIH (3 Normal, 3 Pneumonia)
nih_normal_idx = np.where(y_nih == 0)[0][:3]
nih_pneum_idx  = np.where(y_nih == 1)[0][:3]
for i, idx in enumerate(list(nih_normal_idx) + list(nih_pneum_idx)):
    axes[1, i].imshow(X_nih[idx], cmap='gray')
    axes[1, i].set_title('Normal' if y_nih[idx] == 0 else 'Pneumonia', fontsize=9)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as sample_images.png')

## Step 8 — Save preprocessed arrays for use in later notebooks

In [ ]:
os.makedirs('preprocessed', exist_ok=True)

np.save('preprocessed/X_train.npy', X_train)
np.save('preprocessed/y_train.npy', y_train)
np.save('preprocessed/X_test.npy',  X_test)
np.save('preprocessed/y_test.npy',  y_test)
np.save('preprocessed/X_nih.npy',   X_nih)
np.save('preprocessed/y_nih.npy',   y_nih)

print('All arrays saved to /preprocessed/')
print()
print('Summary:')
print(f'  X_train : {X_train.shape}  — Kaggle training images')
print(f'  y_train : {y_train.shape}')
print(f'  X_test  : {X_test.shape}   — Kaggle test images')
print(f'  y_test  : {y_test.shape}')
print(f'  X_nih   : {X_nih.shape}    — NIH generalization images')
print(f'  y_nih   : {y_nih.shape}')
print()
print('Notebook 1 complete. Open notebook 2 (K-Means segmentation) next.')